<a href="https://colab.research.google.com/github/Sanat1427/USL-LAB/blob/main/q3_05_02_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from sklearn.datasets import fetch_20newsgroups
import math
import random

data = fetch_20newsgroups(
    subset='all',
    remove=('headers', 'footers', 'quotes')
)

documents = data.data[:200]
print("Documents loaded:", len(documents))


Documents loaded: 200


In [3]:
stopwords = ["the","is","and","to","of","in","that","it","on","for","with","as","was","were","be","by"]

def preprocess(doc):
    doc = doc.lower()
    words = doc.split()
    return [w for w in words if w.isalpha() and w not in stopwords]

processed_docs = [preprocess(doc) for doc in documents]


In [4]:
vocab = []

for doc in processed_docs:
    for word in doc:
        if word not in vocab:
            vocab.append(word)

print("Vocabulary size:", len(vocab))


Vocabulary size: 4416


In [5]:
def compute_tf(doc, vocab):
    tf = {}
    doc_len = len(doc)

    for word in vocab:
        tf[word] = doc.count(word) / doc_len if doc_len != 0 else 0

    return tf

tf_docs = [compute_tf(doc, vocab) for doc in processed_docs]


In [6]:
def compute_idf(docs, vocab):
    idf = {}
    N = len(docs)

    for word in vocab:
        count = 0
        for doc in docs:
            if word in doc:
                count += 1
        idf[word] = math.log(N / (1 + count))

    return idf

idf = compute_idf(processed_docs, vocab)


In [7]:
def compute_tfidf(tf, idf):
    tfidf = {}
    for word in idf:
        tfidf[word] = tf[word] * idf[word]
    return tfidf

tfidf_docs = [compute_tfidf(tf, idf) for tf in tf_docs]

print("TF-IDF docs created:", len(tfidf_docs))


TF-IDF docs created: 200


In [8]:
def vectorize(tfidf, vocab):
    return [tfidf[word] for word in vocab]

vectors = [vectorize(doc, vocab) for doc in tfidf_docs]

print("Vectors created:", len(vectors))


Vectors created: 200


In [9]:
K = 5
iterations = 5

centroids = random.sample(vectors, K)

for _ in range(iterations):
    clusters = [[] for _ in range(K)]

    for vec in vectors:
        distances = [math.sqrt(sum((a-b)**2 for a,b in zip(vec,c))) for c in centroids]
        idx = distances.index(min(distances))
        clusters[idx].append(vec)

    new_centroids = []
    for cluster in clusters:
        centroid = [sum(values)/len(values) for values in zip(*cluster)] if cluster else random.choice(vectors)
        new_centroids.append(centroid)

    centroids = new_centroids


In [10]:
def silhouette_score(vectors, clusters):
    scores = []

    for vec in vectors:
        own = next(c for c in clusters if vec in c)

        a = sum(math.sqrt(sum((a-b)**2 for a,b in zip(vec,v))) for v in own) / len(own)

        b = min(
            sum(math.sqrt(sum((a-b)**2 for a,b in zip(vec,v))) for v in c) / len(c)
            for c in clusters if c != own and c
        )

        scores.append((b - a) / max(a, b))

    return sum(scores) / len(scores)

print("Silhouette Score:", silhouette_score(vectors, clusters))


Silhouette Score: -0.15288084306712238


In [12]:
def euclidean(v1, v2):
    return sum((a - b) ** 2 for a, b in zip(v1, v2)) ** 0.5


In [13]:
K = 5
iterations = 5

centroids = random.sample(vectors, K)

for _ in range(iterations):
    clusters = [[] for _ in range(K)]

    # Assignment step
    for vec in vectors:
        distances = [euclidean(vec, c) for c in centroids]
        cluster_index = distances.index(min(distances))
        clusters[cluster_index].append(vec)

    # Update step
    new_centroids = []
    for cluster in clusters:
        if cluster:
            centroid = [sum(values)/len(values) for values in zip(*cluster)]
            new_centroids.append(centroid)
        else:
            new_centroids.append(random.choice(vectors))

    centroids = new_centroids


In [14]:
for i, cluster in enumerate(clusters):
    print("\nCluster", i)
    avg = [sum(v)/len(v) for v in zip(*cluster)]
    top = sorted(range(len(avg)), key=lambda x: avg[x], reverse=True)[:10]
    for idx in top:
        print(vocab[idx], end=" ")
    print()



Cluster 0
news good herb feverfew preventing prevent migraines risks blurb side 

Cluster 1
metal me misconduct compilation oyster judus blessings disguise slayer awaits 

Cluster 2
netmask changing try similar problem had or i a am 

Cluster 3
maximum digital ohm powered static max bet resistance slow versions 

Cluster 4
i a subscrive you they have please up or your 
